# 🏫 Smart Classroom AI — Train From Scratch (Google Drive Version)

**No uploads needed!** This notebook reads your videos directly from Google Drive.

| Step | What Happens |
|------|--------------|
| 1 | Install & import tools |
| 2 | Connect Google Drive |
| 3 | Tell us where your videos are |
| 4 | Extract frames from videos automatically |
| 5 | Prepare images for training |
| 6 | Build custom CNN from scratch |
| 7 | Train the model |
| 8 | Draw training charts |
| 9 | Confusion matrix (exam report) |
| 10 | Save & download model |

---
### ⚡ First: Enable Free GPU
**Runtime → Change Runtime Type → T4 GPU → Save**

In [ ]:
# ═══════════════════════════════════════════════════════════
# STEP 1: Import tools
# Like opening your toolbox before starting work
# ═══════════════════════════════════════════════════════════

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
from torchvision import datasets
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
import numpy as np
import os, json, cv2
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
from PIL import Image

# Use GPU if available (10x faster training - FREE on Colab!)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)} — Training will be FAST!')
else:
    print('No GPU found! Go to Runtime → Change Runtime Type → T4 GPU')

print('\n✅ All tools loaded!')

---
## 📂 STEP 2: Connect Your Google Drive

Run the cell below. It will:
1. Show a Google sign-in popup
2. Ask you to allow access
3. Connect your Drive — your videos become visible to this notebook!

**Your files will be readable at:** `/content/drive/MyDrive/`

In [ ]:
# ═══════════════════════════════════════════════════════════
# STEP 2: Mount Google Drive
# This connects your Google Drive so we can read your videos
# ═══════════════════════════════════════════════════════════

from google.colab import drive
drive.mount('/content/drive')

print('\n✅ Google Drive connected!')
print('Your Drive files are available at: /content/drive/MyDrive/')
print()

# Show top-level contents of your Drive so you can find your videos
print('📁 Contents of your Google Drive root folder:')
try:
    items = os.listdir('/content/drive/MyDrive/')
    for item in sorted(items):
        full = os.path.join('/content/drive/MyDrive/', item)
        kind = '📁' if os.path.isdir(full) else '📄'
        print(f'  {kind} {item}')
except Exception as e:
    print(f'Error listing drive: {e}')

---
## 🔍 STEP 3: Find Your Video Paths

Run the cell below to **see all video files inside a specific folder** on your Drive.
Change `FOLDER_TO_BROWSE` to the folder where your videos are stored.

In [ ]:
# ═══════════════════════════════════════════════════════════
# STEP 3: Browse your Drive to find video paths
# Change the folder name below to match your Drive structure
# ═══════════════════════════════════════════════════════════

# ↓↓↓ CHANGE THIS to the folder where your videos are ↓↓↓
FOLDER_TO_BROWSE = '/content/drive/MyDrive/'
# Examples:
# FOLDER_TO_BROWSE = '/content/drive/MyDrive/Project/'
# FOLDER_TO_BROWSE = '/content/drive/MyDrive/ICT/videos/'
# FOLDER_TO_BROWSE = '/content/drive/MyDrive/classroom_videos/'

print(f'Browsing: {FOLDER_TO_BROWSE}')
print('─' * 60)

video_extensions = ('.mp4', '.avi', '.mov', '.mkv', '.MP4', '.AVI', '.MOV')

for root, dirs, files in os.walk(FOLDER_TO_BROWSE):
    level = root.replace(FOLDER_TO_BROWSE, '').count(os.sep)
    indent = '  ' * level
    folder_name = os.path.basename(root)
    if level == 0:
        print(f'📁 {root}')
    else:
        print(f'{indent}📁 {folder_name}/')
    for file in files:
        sub = '  ' * (level + 1)
        if file.endswith(video_extensions):
            full_path = os.path.join(root, file)
            size_mb = os.path.getsize(full_path) / 1024 / 1024
            print(f'{sub}🎥 {file}  ({size_mb:.1f} MB)')
            print(f'{sub}   Full path: {full_path}')
        else:
            print(f'{sub}📄 {file}')

print('\n✅ Copy the "Full path" lines above into STEP 4 below!')

---
## ⚙️ STEP 4: Tell Us Where Your Videos Are

Edit the `video_map` dictionary below:
- On the **left** side: paste the full path to your video (from Step 3)
- On the **right** side: type `low`, `medium`, or `high`

```
LOW    = classroom with 1-2 people
MEDIUM = classroom with 3-9 people
HIGH   = classroom with 10+ people
```

In [ ]:
# ═══════════════════════════════════════════════════════════
# STEP 4: ✏️  EDIT THIS SECTION — Add your video paths here
# ═══════════════════════════════════════════════════════════

video_map = {
    # ─────────────────────────────────────────────────────────
    # FORMAT:  'full path to video'  :  'label'
    # LABELS:  low  /  medium  /  high
    # ─────────────────────────────────────────────────────────

    '/content/drive/MyDrive/YOUR_VIDEO_LOW.mp4'    : 'low',
    '/content/drive/MyDrive/YOUR_VIDEO_MEDIUM.mp4' : 'medium',
    '/content/drive/MyDrive/YOUR_VIDEO_HIGH.mp4'   : 'high',

    # You can add MORE videos for the same label — more = better AI!
    # '/content/drive/MyDrive/low_video2.mp4'  : 'low',
    # '/content/drive/MyDrive/high_video2.mp4' : 'high',
}

# How many frames to SKIP between each saved frame
# every_n=10 means: save 1 frame, skip 9 frames, save 1 frame, skip 9...
# Lower number = more images (better for training, slower extraction)
EVERY_N_FRAMES = 10

print('✅ Video configuration saved!')
print(f'   Videos configured: {len(video_map)}')
print(f'   Extracting 1 frame every {EVERY_N_FRAMES} frames')
print()
print('Checking if files exist...')
all_ok = True
for path, label in video_map.items():
    exists = os.path.exists(path)
    status = '✅ Found' if exists else '❌ NOT FOUND — check the path!'
    print(f'  [{label.upper():6}] {os.path.basename(path)} → {status}')
    if not exists:
        all_ok = False

if all_ok:
    print('\n✅ All videos found! Run the next cell to extract frames.')
else:
    print('\n❌ Some videos not found. Fix the paths above and run this cell again.')

---
## 🎬 STEP 5: Extract Frames From Videos

A video is just many photos shown quickly (like a flipbook 📖).
We extract 1 photo every 10 frames and save them into folders:
```
dataset/
  low/      ← frames from your LOW videos
  medium/   ← frames from your MEDIUM videos
  high/     ← frames from your HIGH videos
```
These photos become our **training dataset** — the study material for the AI!

In [ ]:
# ═══════════════════════════════════════════════════════════
# STEP 5: Extract frames from all videos
# This reads your videos from Drive and saves frames locally
# ═══════════════════════════════════════════════════════════

def extract_frames(video_path, output_dir, label, every_n=10):
    """
    Opens a video file and saves every Nth frame as a .jpg image.
    Like cutting a flipbook into individual pages.
    """
    save_dir = os.path.join(output_dir, label)
    os.makedirs(save_dir, exist_ok=True)

    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        print(f'  ❌ Could not open: {video_path}')
        return 0

    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps          = cap.get(cv2.CAP_PROP_FPS)
    duration_s   = total_frames / fps if fps > 0 else 0

    print(f'  🎥 {os.path.basename(video_path)}')
    print(f'     Duration: {duration_s:.0f}s | Frames: {total_frames} | FPS: {fps:.1f}')

    count, saved = 0, 0
    # Count existing images so we don't overwrite if re-running
    existing = len([f for f in os.listdir(save_dir) if f.endswith('.jpg')])

    while True:
        ret, frame = cap.read()
        if not ret:
            break
        if count % every_n == 0:
            frame_resized = cv2.resize(frame, (224, 224))
            out_path = os.path.join(save_dir, f'{label}_{existing+saved:05d}.jpg')
            cv2.imwrite(out_path, frame_resized)
            saved += 1
        count += 1

    cap.release()
    print(f'     ✅ Extracted {saved} frames → dataset/{label}/')
    return saved


# Create dataset folder
dataset_path = 'dataset'
os.makedirs(dataset_path, exist_ok=True)

print('🎬 Extracting frames from all videos...')
print('=' * 55)

total_extracted = 0
for video_path, label in video_map.items():
    if os.path.exists(video_path):
        n = extract_frames(video_path, dataset_path, label, every_n=EVERY_N_FRAMES)
        total_extracted += n
    else:
        print(f'  ❌ Skipping (not found): {video_path}')

print('=' * 55)
print(f'\n📊 Dataset Summary:')
grand_total = 0
for cls in ['low', 'medium', 'high']:
    p = os.path.join(dataset_path, cls)
    n = len([f for f in os.listdir(p) if f.endswith('.jpg')]) if os.path.exists(p) else 0
    grand_total += n
    note = '✅' if n >= 100 else '⚠️  Try to get at least 100 images per class'
    print(f'  {cls.upper():8}: {n:4d} images  {note}')
print(f'  {"─"*35}')
print(f'  TOTAL   : {grand_total} images')

---
## 🔧 STEP 6: Prepare Images for Training

AI needs images in exactly the right format.
We also use **Data Augmentation** — artificially making more training images by:
- Flipping images left-right 🔄
- Rotating slightly 🔃
- Changing brightness ☀️

200 real photos → effectively 2000+ training examples!

In [ ]:
# ═══════════════════════════════════════════════════════════
# STEP 6: Set up image preparation and augmentation
# ═══════════════════════════════════════════════════════════

IMAGE_SIZE = 224   # All images resized to 224x224 pixels
BATCH_SIZE = 32    # Process 32 images at a time

# Training transforms — WITH augmentation
# (Used when teaching the model)
train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),                            # Flip image
    transforms.RandomRotation(15),                                      # Tilt slightly
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2), # Lighting
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),        # Zoom
    transforms.ToTensor(),                                              # Convert to numbers
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) # Standardize
])

# Validation transforms — NO augmentation (fair test)
val_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load all images from dataset/ folder automatically
# It reads folders: low/ medium/ high/ and assigns labels automatically
full_dataset = datasets.ImageFolder(root=dataset_path, transform=train_transforms)
class_names  = full_dataset.classes  # e.g. ['high', 'low', 'medium']
num_classes  = len(class_names)

print(f'Classes found: {class_names}')
print(f'Total images:  {len(full_dataset)}')

# Split: 80% training, 20% validation
# Like: 80% of questions for studying, 20% saved for final exam
train_size = int(0.8 * len(full_dataset))
val_size   = len(full_dataset) - train_size
train_ds, val_ds = random_split(full_dataset, [train_size, val_size])
val_ds.dataset.transform = val_transforms  # No augmentation on exam data

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

print(f'\nTraining images:   {train_size}')
print(f'Validation images: {val_size}')
print(f'Batch size:        {BATCH_SIZE}')
print('\n✅ Data ready for training!')

---
## 🧠 STEP 7: Build Our AI Brain From Scratch

We build a **CNN (Convolutional Neural Network)** — it works like eyes + brain:
```
Image
  → Eye Layer 1  (sees basic edges)
  → Eye Layer 2  (sees shapes)
  → Eye Layer 3  (sees objects like chairs, people)
  → Eye Layer 4  (sees crowd density)
  → Brain Layer  (decides: LOW / MEDIUM / HIGH)
```

Every single number starts at **zero** — the AI knows **nothing** at the start.
It learns everything only from YOUR classroom videos! (No YOLO, No pretrained weights)

In [ ]:
# ═══════════════════════════════════════════════════════════
# STEP 7: Build Custom CNN from scratch
# NO pretrained weights — learns only from your videos!
# ═══════════════════════════════════════════════════════════

class ClassroomCNN(nn.Module):
    """
    Custom CNN built from scratch.
    Input : 224x224 colour photo
    Output: 3 confidence scores (one per class: LOW, MEDIUM, HIGH)
    """
    def __init__(self, num_classes=3):
        super().__init__()

        # EYES — scan the image layer by layer
        self.features = nn.Sequential(

            # Eye 1: Detect basic edges  (224 → 112 pixels)
            nn.Conv2d(3, 32, kernel_size=3, padding=1),  # 3 colour channels → 32 filters
            nn.BatchNorm2d(32),   # Keep numbers stable during training
            nn.ReLU(),            # Only pass positive signals
            nn.MaxPool2d(2, 2),   # Shrink image by half
            nn.Dropout2d(0.1),    # Turn off 10% randomly (prevents memorising)

            # Eye 2: Detect shapes  (112 → 56 pixels)
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.1),

            # Eye 3: Detect objects like chairs, people  (56 → 28 pixels)
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.2),

            # Eye 4: Detect crowd density patterns  (28 → 14 pixels)
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(),
            nn.MaxPool2d(2, 2),
            nn.Dropout2d(0.2),
        )

        # Summarise 14x14 map to 1 number per filter
        self.gap = nn.AdaptiveAvgPool2d((1, 1))

        # BRAIN — make the final decision
        self.brain = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256, 512),
            nn.ReLU(),
            nn.Dropout(0.5),   # Drop 50% during training (prevents cheating!)
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)  # Final answer: score per class
        )

    def forward(self, x):
        x = self.features(x)  # Look at the image
        x = self.gap(x)       # Summarise
        x = self.brain(x)     # Decide
        return x


# Create model and move it to GPU
model = ClassroomCNN(num_classes=num_classes).to(device)

total_params = sum(p.numel() for p in model.parameters())
print('🧠 Custom CNN built from scratch!')
print(f'   Total parameters: {total_params:,}')
print(f'   All {total_params:,} values start at random and will be')
print(f'   learned ONLY from your {len(full_dataset)} classroom images.')
print('   No YOLO. No pretrained weights. 100% original.')

---
## 🎓 STEP 8: Train the Model (Teach the AI)

This is the main learning step. Here's what happens each **epoch** (study session):

```
1. Show AI a batch of 32 photos
2. AI guesses: LOW / MEDIUM / HIGH
3. Check if guess is right → calculate LOSS (how wrong?)
4. Adjust the brain to do better
5. Repeat for ALL photos
6. Test on validation photos (no learning — just exam)
7. Repeat for 30 epochs total
```

Watch the **accuracy go UP** and **loss go DOWN** each epoch!

In [ ]:
# ═══════════════════════════════════════════════════════════
# STEP 8: Train the model
# ═══════════════════════════════════════════════════════════

criterion  = nn.CrossEntropyLoss()   # Measures how WRONG the AI is
optimizer  = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
scheduler  = optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)

NUM_EPOCHS = 30
train_losses, val_losses = [], []
train_accs,   val_accs   = [], []
best_acc   = 0.0
best_state = None

print(f'Starting training — {NUM_EPOCHS} epochs')
print('Each row = one full study session through all your photos')
print('=' * 72)

for epoch in range(NUM_EPOCHS):

    # ── Training phase (learning ON) ───────────────────────────────────
    model.train()
    t_loss, t_correct, t_total = 0.0, 0, 0

    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()                    # Clear old corrections
        outputs = model(imgs)                    # AI makes a guess
        loss    = criterion(outputs, labels)     # How wrong was the guess?
        loss.backward()                          # Find what caused the error
        optimizer.step()                         # Fix the brain
        t_loss    += loss.item()
        t_correct += (outputs.argmax(1) == labels).sum().item()
        t_total   += labels.size(0)

    # ── Validation phase (exam — learning OFF) ─────────────────────────
    model.eval()
    v_loss, v_correct, v_total = 0.0, 0, 0

    with torch.no_grad():   # Do NOT adjust brain during exam
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            out    = model(imgs)
            v_loss += criterion(out, labels).item()
            v_correct += (out.argmax(1) == labels).sum().item()
            v_total   += labels.size(0)

    # Calculate results for this epoch
    ta = 100.0 * t_correct / t_total
    va = 100.0 * v_correct / v_total
    tl = t_loss / len(train_loader)
    vl = v_loss / len(val_loader)

    train_losses.append(tl); val_losses.append(vl)
    train_accs.append(ta);   val_accs.append(va)
    scheduler.step()

    # Save the best model
    if va > best_acc:
        best_acc   = va
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
        tag = '  ⭐ NEW BEST SAVED'
    else:
        tag = ''

    print(f'Epoch {epoch+1:2d}/{NUM_EPOCHS} | '
          f'Train Loss: {tl:.4f}  Acc: {ta:5.1f}% | '
          f'Val Loss: {vl:.4f}  Acc: {va:5.1f}%{tag}')

print('\n' + '=' * 72)
print(f'🎉 Training complete!  Best Validation Accuracy: {best_acc:.1f}%')

In [ ]:
# ═══════════════════════════════════════════════════════════
# STEP 9: Draw training progress charts
# Good signs: accuracy UP, loss DOWN, both lines close together
# ═══════════════════════════════════════════════════════════

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
epochs_x = range(1, NUM_EPOCHS + 1)

ax1.plot(epochs_x, train_losses, 'b-o', markersize=3, label='Training Loss')
ax1.plot(epochs_x, val_losses,   'r-o', markersize=3, label='Validation Loss')
ax1.set_title('Loss per Epoch\n(Lower = AI making fewer mistakes)', fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(epochs_x, train_accs, 'b-o', markersize=3, label='Training Accuracy')
ax2.plot(epochs_x, val_accs,   'r-o', markersize=3, label='Validation Accuracy')
ax2.set_title('Accuracy per Epoch\n(Higher = AI getting more answers right)', fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy %')
ax2.set_ylim(0, 100)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Smart Classroom CNN — Training Results', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'Chart saved! Final results:')
print(f'  Train accuracy : {train_accs[-1]:.1f}%')
print(f'  Val accuracy   : {val_accs[-1]:.1f}%')
print(f'  Best val acc   : {best_acc:.1f}%')

---
## 📊 STEP 10: Confusion Matrix — Where Does Our AI Get Confused?

**Diagonal numbers** = correct predictions ✅
**Off-diagonal numbers** = mistakes ❌

Example:
```
             Predicted:
            LOW  MED  HIGH
Actual LOW  [95]  [5]  [0]   → 95% correct!
       MED  [3]  [90]  [7]   → 90% correct
       HIGH [0]   [2]  [98]  → 98% correct
```
Aim for **>80%** on all diagonal cells!

In [ ]:
# ═══════════════════════════════════════════════════════════
# STEP 10: Confusion Matrix + Classification Report
# ═══════════════════════════════════════════════════════════

# Load the best model we saved during training
model.load_state_dict(best_state)
model.eval()

all_preds, all_true = [], []
with torch.no_grad():
    for imgs, labels in val_loader:
        imgs = imgs.to(device)
        preds = model(imgs).argmax(1)
        all_preds.extend(preds.cpu().numpy())
        all_true.extend(labels.numpy())

# Build confusion matrix
cm     = confusion_matrix(all_true, all_preds)
cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100

plt.figure(figsize=(8, 6))
sns.heatmap(cm_pct, annot=True, fmt='.1f', cmap='Blues',
            xticklabels=[c.upper() for c in class_names],
            yticklabels=[c.upper() for c in class_names],
            cbar_kws={'label': 'Accuracy %'})
plt.title('Confusion Matrix (%)\nDiagonal = correct | Off-diagonal = mistakes',
          fontsize=12, fontweight='bold')
plt.ylabel('Actual Label')
plt.xlabel('AI Predicted Label')
plt.tight_layout()
plt.savefig('confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nDetailed Classification Report:')
print('  precision = when AI says X, how often is it really X?')
print('  recall    = out of all real X, how many did AI catch?')
print('  f1-score  = balance of both (aim for > 0.80)')
print()
print(classification_report(
    all_true, all_preds,
    target_names=[c.upper() for c in class_names]
))

---
## 💾 STEP 11: Export & Download Model

We export in **ONNX format** — a universal file that works inside Docker without needing PyTorch installed.

Think of it like converting a Word document to PDF — the PDF opens anywhere!

After downloading, copy these files to your project:
```
classroom_occupancy.onnx  →  edge_app/model/
class_config.json         →  edge_app/model/
```

In [ ]:
# ═══════════════════════════════════════════════════════════
# STEP 11: Export model to ONNX and download everything
# ═══════════════════════════════════════════════════════════

from google.colab import files as colab_files

model.load_state_dict(best_state)
model.eval()

# ── Save PyTorch format (.pth) — for retraining later ──────
torch.save({
    'model_state_dict' : best_state,
    'class_names'      : class_names,
    'image_size'       : IMAGE_SIZE,
    'best_val_accuracy': best_acc
}, 'classroom_occupancy_model.pth')
print('Saved: classroom_occupancy_model.pth')

# ── Export ONNX format (.onnx) — for Docker deployment ──────
dummy_input = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE).to(device)
torch.onnx.export(
    model, dummy_input, 'classroom_occupancy.onnx',
    export_params=True, opset_version=11,
    input_names=['image_input'], output_names=['occupancy_scores'],
    dynamic_axes={'image_input': {0: 'batch'}, 'occupancy_scores': {0: 'batch'}}
)
print('Saved: classroom_occupancy.onnx')

# ── Save class config (.json) ────────────────────────────────
with open('class_config.json', 'w') as f:
    json.dump({
        'class_names'  : class_names,
        'image_size'   : IMAGE_SIZE,
        'best_accuracy': best_acc,
        'model_info'   : 'Custom CNN from scratch — no pretrained weights'
    }, f, indent=2)
print('Saved: class_config.json')

# ── Optionally save to Google Drive too ─────────────────────
drive_save_dir = '/content/drive/MyDrive/smart_classroom_model/'
os.makedirs(drive_save_dir, exist_ok=True)
for fname in ['classroom_occupancy.onnx', 'classroom_occupancy_model.pth',
              'class_config.json', 'training_curves.png', 'confusion_matrix.png']:
    if os.path.exists(fname):
        import shutil
        shutil.copy(fname, os.path.join(drive_save_dir, fname))
print(f'\n✅ Also saved all files to Google Drive: {drive_save_dir}')

# ── Download to your computer ─────────────────────────────────
print('\n📥 Downloading files to your computer...')
for fname in ['classroom_occupancy.onnx', 'classroom_occupancy_model.pth',
              'class_config.json', 'training_curves.png', 'confusion_matrix.png']:
    if os.path.exists(fname):
        size_mb = os.path.getsize(fname) / 1024 / 1024
        colab_files.download(fname)
        print(f'  Downloaded: {fname}  ({size_mb:.1f} MB)')

print()
print('=' * 55)
print(f'TRAINING COMPLETE!')
print(f'Best Accuracy: {best_acc:.1f}%')
print()
print('Next steps:')
print('  1. Copy classroom_occupancy.onnx  →  edge_app/model/')
print('  2. Copy class_config.json         →  edge_app/model/')
print('  3. Run: docker-compose up')
print('  4. Open: http://localhost:5000')